In [1]:
import numpy as np
import pandas as pd
from torch.utils.data.dataset import Dataset
import sklearn

# General Data Initialization

In [2]:
### GLOBAL PATH INPUTS

pData = 'data/'

pAl          = pData + 'Al/'
pAK          = pAl + 'AK/'
pUTdisNodes  = pAK + 'Ductile-disNodes-FCC-12X16/'
pUTdisNodes2 = pAK + '20_RD02_10mm/'
pUTdisStruts = pAK + 'Ductile-disStruts-FCC-12X16/'
pFTdisNodes  = pAK + 'Fracture-disNodes/'

In [3]:
PATH  = pUTdisNodes2
dis   = 'dN'

CSV_train_in  = PATH + f'NN-UT-{dis}-trainIN.csv'
CSV_train_out = PATH + f'NN-UT-{dis}-trainOUT.csv'
CSV_val_in  = PATH + f'NN-UT-{dis}-valIN.csv'
CSV_val_out = PATH + f'NN-UT-{dis}-valOUT.csv'
CSV_test_in  = PATH + f'NN-UT-{dis}-testIN.csv'
CSV_test_out = PATH + f'NN-UT-{dis}-testOUT.csv'

INcsv = PATH + f'Ductile-disNodes-IN.csv'
OUTcsv = PATH + f'Ductile-disNodes-OUT.csv'

In [4]:
def load_TrainTestData(CSV_train_in, CSV_train_out, CSV_val_in, CSV_val_out, CSV_test_in, CSV_test_out):
    train_in = pd.read_csv(CSV_train_in, index_col=0, header=0).to_numpy()
    train_out = pd.read_csv(CSV_train_out, index_col=0, header=0).to_numpy()
    val_in = pd.read_csv(CSV_val_in, index_col=0, header=0).to_numpy()
    val_out = pd.read_csv(CSV_val_out, index_col=0, header=0).to_numpy()
    test_in = pd.read_csv(CSV_test_in, index_col=0, header=0).to_numpy()
    test_out = pd.read_csv(CSV_test_out, index_col=0, header=0).to_numpy()
    return train_in, train_out, val_in, val_out, test_in, test_out

def load_perData(INcsv, OUTcsv):
    IN_df = pd.read_csv(INcsv, index_col=0).sort_index()
    OUT_df = pd.read_csv(OUTcsv, index_col=0).sort_index()
    perIN_df = IN_df.loc[:0]
    perOUT_df = OUT_df.loc[:0]
    return perIN_df.to_numpy()[0], perOUT_df.to_numpy()[:,1:]

In [5]:
def dataParams(x):
    return [np.min(x), np.max(x), np.mean(x), np.std(x)]

def standardize(x, minx, maxx, mode=0):
    if mode == 0:
        return (x - minx)/(maxx - minx)
    if mode == 1:
        return (x*(maxx - minx)) + minx
    
def normalize(x, mean, std, mode=0):
    if mode == 0:
        return (x - mean)/std
    if mode == 1:
        return (x*std) + mean

class Dataset_(Dataset):
    def __init__(self, x, y, edge_index=None):
        self.x = x
        self.y = y
        self.edge_index=edge_index
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.x.shape[0]

In [6]:
def __initData__():
    train_in, train_out, val_in, val_out, test_in, test_out = load_TrainTestData(CSV_train_in, 
                                                                                 CSV_train_out, 
                                                                                 CSV_val_in, 
                                                                                 CSV_val_out, 
                                                                                 CSV_test_in, 
                                                                                 CSV_test_out)
    perIN, perOUT = load_perData(INcsv, OUTcsv)

    inParams = dataParams(np.concatenate((train_in, val_in, test_in)))
    outParams = dataParams(np.concatenate((train_out, val_out, test_out)))

    train_inST = standardize(train_in, inParams[0], inParams[1])
    train_outST = standardize(train_out, outParams[0], outParams[1])
    val_inST = standardize(val_in, inParams[0], inParams[1])
    val_outST = standardize(val_out, outParams[0], outParams[1])
    test_inST = standardize(test_in, inParams[0], inParams[1])
    test_outST = standardize(test_out, outParams[0], outParams[1])
    
    train_inNM = normalize(train_in, inParams[2], inParams[3])
    train_outNM = normalize(train_out, outParams[2], outParams[3])
    val_inNM = normalize(val_in, inParams[2], inParams[3])
    val_outNM = normalize(val_out, outParams[2], outParams[3])
    test_inNM = normalize(test_in, inParams[2], inParams[3])
    test_outNM = normalize(test_out, outParams[2], outParams[3])
    
    r1 = [train_in, train_out, val_in, val_out, test_in, test_out]
    r2 = [perIN, perOUT]
    r3 = [inParams, outParams]
    r4 = [train_inST, train_outST, val_inST, val_outST, test_inST, test_outST]
    r5 = [train_inNM, train_outNM, val_inNM, val_outNM, test_inNM, test_outNM]
    return r1, r2, r3, r4, r5

In [7]:
gi1, gi2, gi3, gi4, gi5 = __initData__()

train_in, train_out, val_in, val_out, test_in, test_out = gi1
perIN, perOUT = gi2
inParams, outParams = gi3
train_inST, train_outST, val_inST, val_outST, test_inST, test_outST = gi4
train_inNM, train_outNM, val_inNM, val_outNM, test_inNM, test_outNM = gi5

# Distribution Function Optimization - Data Formatting
Unique function approximations from perfect lattice coorindates to each disorder ($\Delta$) distribution.

In [8]:
def DisorderDistribution__initData__(perIN, train_in):
    train_in1 = perIN.reshape(int(len(perIN)/2), 2)
    train_in1 = np.array([i for i in train_in1 if max(train_in1[:,0]) != i[0] and min(train_in1[:,0]) != i[0] and 
                                                  max(train_in1[:,1]) != i[1] and min(train_in1[:,1]) != i[1]])
    #train_in1 = train_in1.flatten()

    train_out1 = train_in.reshape(len(train_in),int(len(train_in[0])/2),2)
    dx_out1 = train_in.reshape(len(train_in),int(len(train_in[0])/2),2)[:,:,0].reshape(len(train_in),int(len(train_in[0])/2),1)
    dy_out1 = train_in.reshape(len(train_in),int(len(train_in[0])/2),2)[:,:,1].reshape(len(train_in),int(len(train_in[0])/2),1)

    inParams1 = dataParams(train_in1)
    train_in1ST = standardize(train_in1, inParams1[0], inParams1[1])
    train_in1NM = normalize(train_in1, inParams1[2], inParams1[3])

    outParams1dx = dataParams(dx_out1)
    outParams1dy = dataParams(dy_out1)
    dx_out1ST = standardize(dx_out1, outParams1dx[0], outParams1dx[1])
    dy_out1ST = standardize(dy_out1, outParams1dy[0], outParams1dy[1])
    dx_out1NM = normalize(dx_out1, outParams1dx[2], outParams1dx[3])
    dy_out1NM = normalize(dy_out1, outParams1dy[2], outParams1dy[3])
    
    r1 = [train_in1, train_out1, dx_out1, dy_out1]
    r2 = [inParams1, outParams1dx, outParams1dy]
    r3 = [train_in1ST, dx_out1ST, dy_out1ST]
    r4 = [train_in1NM, dx_out1NM, dy_out1NM]
    
    return r1, r2, r3, r4

In [9]:
ddi1, ddi2, ddi3, ddi4 = DisorderDistribution__initData__(perIN, train_in)

In [10]:
train_in1, train_out1, dx_out1, dy_out1 = ddi1
inParams1, outParams1dx, outParams1dy = ddi2
train_in1ST, dx_out1ST, dy_out1ST = ddi3
train_in1NM, dx_out1NM, dy_out1NM = ddi4